# 🐍 Day 6: Context Managers

- __enter__ and __exit__
- contextlib module
- @contextmanager decorator
- ExitStack

## What is a Context Manager?

A context manager sets up and tears down resources around a block. Used with `with` statement. Ensures cleanup even if an exception occurs.

In [ ]:
with open('file.txt') as f:
    content = f.read()
# File is closed automatically

## Protocol: __enter__ and __exit__

A class is a context manager if it implements `__enter__` and `__exit__`. `__enter__` returns the resource; `__exit__` handles cleanup.

In [ ]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False  # Don't suppress exceptions

with Timer() as t:
    total = sum(range(1000000))
print(f"Result: {total}")

## __exit__ Parameters

`__exit__(self, exc_type, exc_val, exc_tb)`: If exception occurred, these hold the exception info. Return True to suppress the exception.

In [ ]:
class Suppress:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"Suppressed: {exc_val}")
        return True  # Suppress exception

with Suppress():
    raise ValueError("Oops!")
print("Continuing after suppressed exception")

## contextlib.contextmanager

`@contextmanager` turns a generator into a context manager. Code before `yield` = __enter__; code after = __exit__.

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_resource():
    print("Setup")
    resource = "resource"
    try:
        yield resource
    finally:
        print("Cleanup")

with managed_resource() as r:
    print(f"Using {r}")

## contextlib.suppress

`suppress(*exceptions)` suppresses specified exceptions within the block.

In [ ]:
from contextlib import suppress

with suppress(FileNotFoundError):
    with open('nonexistent.txt') as f:
        print(f.read())
print("Continued after potential FileNotFoundError")

## contextlib.redirect_stdout

Temporarily redirect stdout to another file or stream.

In [ ]:
from contextlib import redirect_stdout
import io

buf = io.StringIO()
with redirect_stdout(buf):
    print("Hello")
    print("World")
print("Captured:", repr(buf.getvalue()))

## ExitStack - Dynamic Context Managers

`ExitStack` manages multiple context managers dynamically. Enter/exit in LIFO order.

In [ ]:
from contextlib import ExitStack

with ExitStack() as stack:
    files = [stack.enter_context(open(f'file{i}.txt', 'w')) for i in range(3)]
    for i, f in enumerate(files):
        f.write(f'content {i}\n')
# All 3 files closed automatically

# Simpler: we may not have those files, so use a safe demo
from contextlib import ExitStack
from io import StringIO

with ExitStack() as stack:
    buffers = [stack.enter_context(StringIO()) for _ in range(2)]
    buffers[0].write('a')
    buffers[1].write('b')
print("ExitStack demo done")

## nullcontext

`nullcontext(enter_result=None)` - no-op context manager. Useful when optionally using a context.

In [ ]:
from contextlib import nullcontext

# When you don't need a real context, use nullcontext
with nullcontext('default') as x:
    print(x)

## Practical: Temporary Directory

Create a temp dir, use it, then clean up.

In [ ]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    print(f"Temp dir: {tmpdir}")
    filepath = os.path.join(tmpdir, 'test.txt')
    with open(filepath, 'w') as f:
        f.write('hello')
    print(os.path.exists(filepath))
print(os.path.exists(tmpdir))  # False - cleaned up

## Common Mistakes

- **Forgetting return False in __exit__**: Return True only if you want to suppress exceptions
- **@contextmanager without try/finally**: If yield raises, cleanup won't run; use try/finally
- **Multiple resources**: Use nested with or ExitStack
- **Yield value**: What you yield is the "as" value; yield nothing for no value

In [ ]:
# WRONG: No try/finally - cleanup skipped if exception in block
# @contextmanager
# def bad():
#     setup()
#     yield
#     cleanup()  # Never runs if exception before yield!

# CORRECT: Use try/finally
from contextlib import contextmanager
@contextmanager
def good():
    print("setup")
    try:
        yield
    finally:
        print("cleanup")

with good():
    print("body")

## Exercises

In [ ]:
# Exercise 1: Create a context manager class that prints 'Enter' and 'Exit'.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use @contextmanager to create a simple 'hello' context that yields and prints 'goodbye' on exit.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Create a context manager that changes directory and restores it on exit. Use os.chdir and os.getcwd.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use suppress(ZeroDivisionError) to safely compute 1/0 and print 'OK' after.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Use redirect_stdout to capture print output to a StringIO. Print the captured string.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Use ExitStack to enter two StringIO contexts. Write to both, then verify cleanup.
# YOUR CODE HERE

## Key Takeaways

1. Context manager: __enter__ (setup) and __exit__ (cleanup)
2. __exit__ returns False to propagate exceptions, True to suppress
3. @contextmanager: code before yield = enter, after = exit; use try/finally
4. suppress, redirect_stdout: built-in context utilities
5. ExitStack for dynamic/runtime context management

## 🔜 Next: Day 7 - Mini-Project Data Pipeline

Tomorrow: Build a data pipeline processor using generators and itertools!